# Optimizing Latency in Voice Agent Pipelines

**Research Notebook | Latency Measurement & Analysis**

Perceived responsiveness is the primary quality signal in voice AI — users tolerate delays of up to ~500ms before a conversation feels unnatural. This notebook instruments each stage of the voice agent pipeline to measure, isolate, and reason about latency contributions.

**Key metrics tracked:**

| Metric | Stage | What it measures |
|---|---|---|
| **EOU Delay** | VAD → STT | Time from end of user speech to transcript commit |
| **Transcription Delay** | STT | Time for Whisper to return the final transcript |
| **TTFT** | LLM | Time-to-first-token: how quickly GPT-4o starts responding |
| **Tokens/s** | LLM | LLM generation throughput |
| **TTFB** | TTS | Time-to-first-byte: how quickly ElevenLabs starts producing audio |

The goal is to identify the **dominant bottleneck** in a given session and inform optimization decisions (e.g., model swap, VAD tuning, streaming configuration).

<div style="background-color:#fff6ff; padding:13px; border-width:3px; border-color:#efe6ef; border-style:solid; border-radius:6px">

<p>💻 &nbsp; <b>View <code>requirements.txt</code>:</b> Click <em>File → Open</em> in the top menu to browse the project directory.</p>

<p>⬇ &nbsp; <b>Download this notebook:</b> Click <em>File → Download as → Notebook (.ipynb)</em>.</p>

<p>🔑 &nbsp; <b>Setup:</b> Ensure <code>OPENAI_API_KEY</code> and <code>ELEVENLABS_API_KEY</code> are set in your <code>.env</code> file before running. See the project <a href="../README.md">README</a> for details.</p>

</div>

<p style="background-color:#f7fff8; padding:15px; border-width:3px; border-color:#e0f0e0; border-style:solid; border-radius:6px"> 🚨
&nbsp; <b>Different Run Results:</b> The output generated by AI models can vary with each execution due to their dynamic, probabilistic nature. Don't be surprised if your results differ from those shown in the video.</p>

## Step 1: Import Modules and Metrics Instrumentation

This notebook extends the base voice agent stack with **`livekit.agents.metrics`** — a set of data classes emitted as events by each pipeline plugin after every conversation turn.

**Standard agent imports** (same as Voice Agent Components):
- `livekit.agents` — `Agent`, `AgentSession`, `JobContext`, `WorkerOptions`, `jupyter`
- `livekit.plugins` — `openai` (STT + LLM), `elevenlabs` (TTS), `silero` (VAD)

**Metrics-specific imports:**

| Class | Emitted by | Key fields |
|---|---|---|
| `LLMMetrics` | `openai.LLM` | `ttft`, `tokens_per_second`, `prompt_tokens`, `completion_tokens` |
| `STTMetrics` | `openai.STT` | `duration`, `audio_duration`, `streamed` |
| `TTSMetrics` | `elevenlabs.TTS` | `ttfb`, `duration`, `audio_duration`, `streamed` |
| `EOUMetrics` | `openai.STT` (EOU event) | `end_of_utterance_delay`, `transcription_delay` |

`asyncio` is imported to schedule metric callbacks as non-blocking async tasks, preventing metric collection from stalling the audio pipeline.

In [ ]:
# Fix asyncio context conflict in Jupyter (RuntimeError: cannot enter context already entered)
import subprocess, sys, warnings
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "nest_asyncio"], check=True)

import nest_asyncio
nest_asyncio.apply()

# Suppress spurious aiohttp/ipykernel warning: "coroutine '.*' was never awaited"
# This is a cosmetic side effect of running an aiohttp app inside the Jupyter kernel — not a real error.
warnings.filterwarnings("ignore", message="coroutine '.*' was never awaited", category=RuntimeWarning)

In [1]:
import logging
import os

from dotenv import load_dotenv
_ = load_dotenv(override=True)

logger = logging.getLogger("dlai-agent")
logger.setLevel(logging.INFO)

from livekit import agents
from livekit.agents import Agent, AgentSession, JobContext, WorkerOptions, jupyter
from livekit.plugins import (
    openai,
    elevenlabs,
    silero,
)

from livekit.agents.metrics import LLMMetrics, STTMetrics, TTSMetrics, EOUMetrics
import asyncio

## High-Level Architecture — Metrics Collection Points

The diagram below shows where metrics events are emitted (dashed lines) relative to the audio pipeline:

![Latency Architecture](https://raw.githubusercontent.com/redtitan1981/AI/main/AI%20research/images/latency_architecture.png)

> Metrics are collected **per turn** and accumulated in `_metrics_summary`. The session summary prints averages across all turns when the agent shuts down.

In [2]:
class MetricsAgent(Agent):
    def __init__(self, model: str = "gpt-4o") -> None:
        llm = openai.LLM(model=model)
        # Switch to "gpt-4o-mini" for lower latency comparison
        stt = openai.STT(model="whisper-1")
        tts = elevenlabs.TTS(api_key=os.getenv("ELEVENLABS_API_KEY"))
        # min_silence_duration_ms=200 reduces CPU usage and avoids "inference slower than realtime" warning
        silero_vad = silero.VAD.load(min_silence_duration_ms=200)

        self._metrics_summary = {"llm": [], "stt": [], "tts": [], "eou": []}

        super().__init__(
            instructions="You are a helpful assistant communicating via voice",
            stt=stt,
            llm=llm,
            tts=tts,
            vad=silero_vad,
        )

        def llm_metrics_wrapper(metrics: LLMMetrics):
            asyncio.create_task(self.on_llm_metrics_collected(metrics))
        llm.on("metrics_collected", llm_metrics_wrapper)

        def stt_metrics_wrapper(metrics: STTMetrics):
            asyncio.create_task(self.on_stt_metrics_collected(metrics))
        stt.on("metrics_collected", stt_metrics_wrapper)

        def eou_metrics_wrapper(metrics: EOUMetrics):
            asyncio.create_task(self.on_eou_metrics_collected(metrics))
        stt.on("eou_metrics_collected", eou_metrics_wrapper)

        def tts_metrics_wrapper(metrics: TTSMetrics):
            asyncio.create_task(self.on_tts_metrics_collected(metrics))
        tts.on("metrics_collected", tts_metrics_wrapper)

    async def on_llm_metrics_collected(self, metrics: LLMMetrics) -> None:
        self._metrics_summary["llm"].append(metrics)
        print("\n--- LLM Metrics ---")
        print(f"Prompt Tokens: {metrics.prompt_tokens}")
        print(f"Completion Tokens: {metrics.completion_tokens}")
        print(f"Tokens per second: {metrics.tokens_per_second:.4f}")
        print(f"TTFT: {metrics.ttft:.4f}s")
        print("------------------\n")

    async def on_stt_metrics_collected(self, metrics: STTMetrics) -> None:
        self._metrics_summary["stt"].append(metrics)
        print("\n--- STT Metrics ---")
        print(f"Duration: {metrics.duration:.4f}s")
        print(f"Audio Duration: {metrics.audio_duration:.4f}s")
        print(f"Streamed: {'Yes' if metrics.streamed else 'No'}")
        print("------------------\n")

    async def on_eou_metrics_collected(self, metrics: EOUMetrics) -> None:
        self._metrics_summary["eou"].append(metrics)
        print("\n--- End of Utterance Metrics ---")
        print(f"End of Utterance Delay: {metrics.end_of_utterance_delay:.4f}s")
        print(f"Transcription Delay: {metrics.transcription_delay:.4f}s")
        print("--------------------------------\n")

    async def on_tts_metrics_collected(self, metrics: TTSMetrics) -> None:
        self._metrics_summary["tts"].append(metrics)
        print("\n--- TTS Metrics ---")
        print(f"TTFB: {metrics.ttfb:.4f}s")
        print(f"Duration: {metrics.duration:.4f}s")
        print(f"Audio Duration: {metrics.audio_duration:.4f}s")
        print(f"Streamed: {'Yes' if metrics.streamed else 'No'}")
        print("------------------\n")

    def print_summary(self) -> None:
        print("\n========== SESSION SUMMARY ==========")
        if self._metrics_summary["llm"]:
            ttfts = [m.ttft for m in self._metrics_summary["llm"]]
            tps = [m.tokens_per_second for m in self._metrics_summary["llm"]]
            print(f"LLM  | Avg TTFT: {sum(ttfts)/len(ttfts):.4f}s  | Avg Tokens/s: {sum(tps)/len(tps):.4f}")
        if self._metrics_summary["tts"]:
            ttfbs = [m.ttfb for m in self._metrics_summary["tts"]]
            print(f"TTS  | Avg TTFB: {sum(ttfbs)/len(ttfbs):.4f}s")
        if self._metrics_summary["eou"]:
            eou_delays = [m.end_of_utterance_delay for m in self._metrics_summary["eou"]]
            print(f"EOU  | Avg Delay: {sum(eou_delays)/len(eou_delays):.4f}s")
        print("=====================================\n")

## Step 2: MetricsAgent — Instrumented Voice Agent

`MetricsAgent` extends `Agent` with per-component latency instrumentation. Unlike the basic `Assistant` in the previous notebook, components (STT, LLM, TTS, VAD) are instantiated **inside the constructor** rather than in `AgentSession` — this is necessary to attach event listeners directly to each plugin instance before the session starts.

**Design pattern — event-driven metrics collection:**

Each plugin emits a `metrics_collected` event after completing a turn. The agent registers a callback for each:

```python
llm.on("metrics_collected", llm_metrics_wrapper)
stt.on("metrics_collected", stt_metrics_wrapper)
stt.on("eou_metrics_collected", eou_metrics_wrapper)   # separate EOU event
tts.on("metrics_collected", tts_metrics_wrapper)
```

Callbacks are wrapped with `asyncio.create_task()` so they run concurrently without blocking the pipeline.

**What each metric tells you:**

| Metric | Interpretation | Optimization lever |
|---|---|---|
| `EOUMetrics.end_of_utterance_delay` | How long VAD takes to detect speech has ended | Lower Silero silence threshold |
| `EOUMetrics.transcription_delay` | Time from speech end to committed transcript | Switch to streaming STT |
| `LLMMetrics.ttft` | Network + model startup time before first token | Use a smaller/faster model (e.g. gpt-4o-mini) |
| `LLMMetrics.tokens_per_second` | LLM generation speed | Model size trade-off |
| `TTSMetrics.ttfb` | Time until first audio byte is synthesized | Use a lower-latency TTS encoding or model |

**`_metrics_summary`** accumulates all per-turn results as lists, enabling `print_summary()` to compute session-level averages for comparative analysis.

In [3]:
async def entrypoint(ctx: JobContext):
    model = os.getenv("LLM_MODEL", "gpt-4o")
    agent = MetricsAgent(model=model)
    try:
        await ctx.connect()
        session = AgentSession()
        await session.start(
            agent=agent,
            room=ctx.room,
        )
    except Exception as e:
        logger.error(f"Agent session failed: {e}")
        raise
    finally:
        agent.print_summary()

## Step 3: Entrypoint with Configurable Model

The entrypoint reads `LLM_MODEL` from the environment, defaulting to `gpt-4o`. This makes it easy to run **A/B latency comparisons** without changing code — for example:

```bash
LLM_MODEL=gpt-4o-mini jupyter nbconvert --to notebook --execute ...
```

**Key design decisions:**
- `MetricsAgent(model=model)` — the model string is forwarded to `openai.LLM()` inside the agent constructor
- `AgentSession()` is instantiated with no arguments because all pipeline components are already bound to the agent
- The `try/finally` block guarantees `agent.print_summary()` runs even if the session errors out mid-conversation, preserving collected metrics

## Step 4: Run and Interpret Metrics

`jupyter.run_app()` starts the instrumented agent in-notebook. The `LIVEKIT_JUPYTER_URL` env var allows overriding the default sandbox token endpoint (useful for connecting to a self-hosted LiveKit server).

**Usage:**
1. Run all cells top-to-bottom first.
2. Unmute the **microphone icon** (bottom-left of the audio widget) to speak.
3. Speak a complete sentence — metrics are printed to the notebook output after each turn.
4. Shut down the cell (Stop button) to trigger `print_summary()` with session averages.

**Reading the per-turn output:**

```
--- End of Utterance Metrics ---
End of Utterance Delay: 0.85s     ← VAD silence detection lag
Transcription Delay:   0.42s      ← Whisper processing time

--- LLM Metrics ---
TTFT:               2.15s         ← Time until GPT-4o first token
Tokens per second:  3.82          ← Generation throughput

--- TTS Metrics ---
TTFB:               1.20s         ← Time until first audio byte
Audio Duration:     1.88s         ← Length of synthesized audio
```

**Rough latency budget for a natural conversation (~500ms target):**

```
Total perceived delay ≈ EOU Delay + Transcription Delay + TTFT + TTFB
                      ≈ 0.85      + 0.42               + 2.15 + 1.20  = ~4.6s  (baseline)
```

> With `gpt-4o-mini` instead of `gpt-4o`, TTFT typically drops to ~0.3–0.8s — the single largest latency saving available without infrastructure changes.

**Expected warnings (non-fatal):**

| Message | Cause | Action |
|---|---|---|
| `inference is slower than realtime` | Silero VAD running on CPU | Tuned via `min_silence_duration_ms=200` — safe to ignore |
| `401 Unauthorized - invalid token` | Sandbox join token expired (~30 min TTL) | Stop the cell and re-run it to get a fresh token |
| `ping timeout` / `pc_state failed` | WebRTC connection dropped after inactivity | Stop the cell and re-run |

In [ ]:
jupyter_url = os.getenv("LIVEKIT_JUPYTER_URL", "https://jupyter-api-livekit.vercel.app/api/join-token")
jupyter.run_app(WorkerOptions(entrypoint_fnc=entrypoint), jupyter_url=jupyter_url)

## Summary & Optimization Strategies

This notebook provides a baseline latency profile for the GPT-4o voice agent pipeline. The session summary gives per-stage averages to identify the dominant bottleneck.

**Optimization strategies by stage:**

| Stage | Technique | Expected impact |
|---|---|---|
| **EOU / VAD** | Reduce Silero silence detection threshold | −100–300ms EOU delay |
| **STT** | Enable streaming transcription | Overlaps transcription with speaking |
| **LLM** | Switch to `gpt-4o-mini` | TTFT drops from ~2s to ~0.3–0.8s |
| **LLM** | Use a self-hosted model (e.g. Llama 3) | Eliminates API round-trip |
| **TTS** | Use `pcm_16000` encoding instead of `mp3_44100` | Lower TTFB, smaller buffer |
| **TTS** | Switch to a lower-latency provider (e.g. Cartesia) | TTFB can reach <100ms |
| **Network** | Co-locate agent worker and API endpoints | Reduces all API latencies |

**Next steps:**
- Run sessions with `LLM_MODEL=gpt-4o-mini` and compare the session summaries
- Plot per-turn TTFT and TTFB over a multi-turn session to see variance
- Combine with the [Voice Agent Components](Voice%20Agent%20Components.ipynb) notebook to apply optimizations to the full pipeline

**References:**
- [LiveKit Agents Metrics API](https://docs.livekit.io/agents/build/metrics/)
- [OpenAI Whisper Streaming](https://platform.openai.com/docs/guides/speech-to-text)
- [Silero VAD Parameters](https://github.com/snakers4/silero-vad/wiki/Quality-Metrics)
- [ElevenLabs Latency Optimization](https://elevenlabs.io/docs/api-reference/text-to-speech)